In [12]:
# CELL 1: IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time
import joblib
import os
warnings.filterwarnings('ignore')

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score,
    matthews_corrcoef, balanced_accuracy_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

# Explainability
import shap

os.makedirs('../results', exist_ok=True)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print(' All libraries loaded!')
print(f'XGBoost version: {xgb.__version__}')

 All libraries loaded!
XGBoost version: 3.2.0


In [13]:
# CELL 2: LOAD DATA

X_train = np.load('../data/X_train.npy')
y_train = np.load('../data/y_train.npy')
X_val   = np.load('../data/X_val.npy')
y_val   = np.load('../data/y_val.npy')
X_test  = np.load('../data/X_test.npy')
y_test  = np.load('../data/y_test.npy')

feature_names = pd.read_csv('../data/feature_names.csv')['feature'].tolist()

print('=== Data Loaded ===')
print(f'X_train: {X_train.shape} | Fraud: {y_train.sum():,} ({y_train.mean()*100:.1f}%) [SMOTE applied]')
print(f'X_val:   {X_val.shape}   | Fraud: {y_val.sum()} ({y_val.mean()*100:.3f}%)')
print(f'X_test:  {X_test.shape}  | Fraud: {y_test.sum()} ({y_test.mean()*100:.3f}%)')
print(f'Features: {len(feature_names)} → {feature_names}')

=== Data Loaded ===
X_train: (200157, 32) | Fraud: 18,196 (9.1%) [SMOTE applied]
X_val:   (45569, 32)   | Fraud: 79 (0.173%)
X_test:  (56962, 32)  | Fraud: 98 (0.172%)
Features: 32 → ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour_sin', 'Hour_cos', 'Log_Amount']


In [14]:
# CELL 3: EVALUATION HELPER FUNCTIONS

def evaluate_model(model, X, y, model_name, threshold=0.5, verbose=True):
    """
    Comprehensive model evaluation for fraud detection.
    Returns dictionary of all metrics.
    """
    # Get probabilities
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    # Core metrics
    auc_roc   = roc_auc_score(y, y_prob)
    avg_prec  = average_precision_score(y, y_prob)
    precision = precision_score(y, y_pred, zero_division=0)
    recall    = recall_score(y, y_pred, zero_division=0)
    f1        = f1_score(y, y_pred, zero_division=0)
    mcc       = matthews_corrcoef(y, y_pred)
    bal_acc   = balanced_accuracy_score(y, y_pred)

    # Confusion matrix values
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    if verbose:
        print(f'\n{"═"*55}')
        print(f'   {model_name} (threshold={threshold:.2f})')
        print(f'{"═"*55}')
        print(f'  AUC-ROC:            {auc_roc:.4f}')
        print(f'  Average Precision:  {avg_prec:.4f}')
        print(f'  Precision:          {precision:.4f}')
        print(f'  Recall (Sensitivity):{recall:.4f}')
        print(f'  F1-Score:           {f1:.4f}')
        print(f'  MCC:                {mcc:.4f}')
        print(f'  Balanced Accuracy:  {bal_acc:.4f}')
        print(f'  False Positive Rate:{fpr:.4f}')
        print(f'  False Negative Rate:{fnr:.4f}')
        print(f'  Confusion Matrix:')
        print(f'    TN={tn:,}  FP={fp}')
        print(f'    FN={fn}   TP={tp}')
        print(f'\n  Classification Report:')
        print(classification_report(y, y_pred,
              target_names=['Legitimate', 'Fraud']))

    return {
        'model_name': model_name,
        'auc_roc': auc_roc, 'avg_precision': avg_prec,
        'precision': precision, 'recall': recall,
        'f1': f1, 'mcc': mcc, 'balanced_accuracy': bal_acc,
        'fpr': fpr, 'fnr': fnr,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'y_prob': y_prob, 'y_pred': y_pred, 'threshold': threshold
    }

results = {}  # Store all model results
print(' Evaluation functions ready!')

 Evaluation functions ready!


In [15]:
# CELL 4: LOGISTIC REGRESSION

print('   Model 1: Logistic Regression (Baseline)')
print('   Why: Simple, interpretable, fast — good baseline')
print('   Algorithm: Logistic function with L2 regularization')
print('-' * 50)

start = time.time()
lr_model = LogisticRegression(
    C=0.01,                   # Regularization (lower = stronger)
    max_iter=1000,             # Sufficient iterations to converge
    class_weight='balanced',  # Handles remaining imbalance
    solver='lbfgs',            # Efficient for large datasets
    random_state=42,
    n_jobs=-1
)
lr_model.fit(X_train, y_train)
lr_time = time.time() - start

print(f'Training time: {lr_time:.2f}s')
results['Logistic Regression'] = evaluate_model(
    lr_model, X_val, y_val, 'Logistic Regression'
)

# Save model
joblib.dump(lr_model, '../results/model_lr.pkl')
print(' Model saved: results/model_lr.pkl')

   Model 1: Logistic Regression (Baseline)
   Why: Simple, interpretable, fast — good baseline
   Algorithm: Logistic function with L2 regularization
--------------------------------------------------
Training time: 6.58s

═══════════════════════════════════════════════════════
   Logistic Regression (threshold=0.50)
═══════════════════════════════════════════════════════
  AUC-ROC:            0.9704
  Average Precision:  0.6822
  Precision:          0.0568
  Recall (Sensitivity):0.8734
  F1-Score:           0.1066
  MCC:                0.2190
  Balanced Accuracy:  0.9241
  False Positive Rate:0.0252
  False Negative Rate:0.1266
  Confusion Matrix:
    TN=44,344  FP=1146
    FN=10   TP=69

  Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.97      0.99     45490
       Fraud       0.06      0.87      0.11        79

    accuracy                           0.97     45569
   macro avg       0.53      0.92      0.55     45569
weig

In [16]:
# CELL 5: CROSS-VALIDATION FOR LOGISTIC REGRESSION

print('=== Cross-Validation: Logistic Regression ===')
print('Using Stratified 5-Fold CV on training data')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_lr = cross_val_score(
    lr_model, X_train, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f'CV AUC-ROC scores: {cv_scores_lr.round(4)}')
print(f'Mean AUC-ROC:      {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}')
print(f'\n Low variance = stable model (no overfitting)')

=== Cross-Validation: Logistic Regression ===
Using Stratified 5-Fold CV on training data
CV AUC-ROC scores: [0.9929 0.9937 0.9925 0.9919 0.9922]
Mean AUC-ROC:      0.9926 ± 0.0006

 Low variance = stable model (no overfitting)
